<a href="https://colab.research.google.com/github/petitoandrea-jpg/freeCodeCamp-Machine-Learning-Python/blob/main/linear_regression/health_costs_calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Download dataset
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
df = pd.read_csv('insurance.csv')

# 1. Converti variabili categoriche in numeriche
df = pd.get_dummies(df, columns=['sex', 'smoker', 'region'], drop_first=True)

# Cast dei boolean a float32 per TF
for col in df.columns:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype('float32')

# 2. Divisione 80% train - 20% test
train_dataset = df.sample(frac=0.8, random_state=0)
test_dataset = df.drop(train_dataset.index)

# 3. Estrazione colonna target ("expenses")
train_labels = train_dataset.pop('expenses')
test_labels = test_dataset.pop('expenses')

In [ ]:
# Normalizzazione delle feature
normalizer = layers.Normalization(axis=-1)
normalizer.adapt(np.array(train_dataset))

# Costruzione modello Keras
model = keras.Sequential([
    normalizer,
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])

# Compilazione con MAE come metrica target
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.1),
    loss='mae',
    metrics=['mae', 'mse']
)

# Addestramento
history = model.fit(
    train_dataset,
    train_labels,
    epochs=100,
    verbose=0,
    validation_split=0.2
)

In [ ]:
# Valutazione del modello sul test set
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
else:
    print("MAE must be under 3500. Keep trying!")

# Plot dei risultati
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True Values [expenses]')
plt.ylabel('Predictions [expenses]')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims, lims)